# Model #

I try to put down a model. I will use the xor-perceptron model as a base and modify that.

first create the lattice, N by N, with Nsol = N**2

I should create a class for the cell object and then the network class is formed by cell objects and when the network is created, each network is associated with a position in the lattice.

Note: for now I'm only considering one possible link between each pair neuron for each direction

In [9]:
import jax
from jax import random as jrd
from jax import numpy as jnp
from jax import debug as jdb

In [10]:
# parameters of the simulation
par = {'key': jrd.key(1634),    # key for random generation
       'N': 10,
       'int_range': 1,             # interaction range
       'p_self_link': 0.5,
       'target_set': [0.0,1.0,1.0,0.0],
       'input_set':[[0,0],[0,1],[1,0],[1,1]]}

In [11]:
dic = {'a': 0,
       'b': 1}

@jax.jit
def up(dic):
    dic['a'] += 2
    return dic

# Pass the dictionary as a tuple of key-value pairs
dic = up(dic)
dic = up(dic)

jdb.print('{dic}', dic=dict(dic))

{'a': Array(4, dtype=int32, weak_type=True), 'b': Array(1, dtype=int32, weak_type=True)}


In [40]:
# define a quick function for generating a new random key from par['key'] and update par['key']
def gen_key(par):
    key, subkey = jrd.split(par['key'])     # generate a random key by splitting the key in par
    par['key'] = key                        # update the key in par
    return subkey, par                      # I have to return also par, otherwise, bc of JIT's rules, par['key'] won't update

# quick function to calculate the Eucledian distance between two sites in the lattice
def cdist(a,b,vmap=False):
    if vmap:
        fun = jax.vmap(lambda a,b: jnp.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2),in_axes=0)
        return fun(a,b)
    return jnp.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)

class Cell(object):
    
    def __init__(self):     
        self.activation = None     # define the activation function through Kolmogorov-Arnold decomposition or Fourier's    # generate a key for random generation 
        self.value = -1    # For now i initialize it to -1, bc I know that can't be 
            
    def generate(self, par):
        self.activation = jax.nn.sigmoid  # DEFINE THIS!!!!
        subkey, par = gen_key(par)                  # generate the subkey for the random generation in the following line
        self.value = jrd.choice(subkey, jnp.arange(2)) # Assign an initial value of either 0 or 1   
    
    def compute(self, input):      # compute the value of the cell given the input and the activation. This will be the value sent to the (eventual) other cells.
        self.value = self.activation(jnp.sum(input))
        

class Network(object):
    
    def __init__(self):             # NON SO SE HA SENSO INIZIALIZZARE COSì LE MATRICI, PERCHè ESSENDO N = 0, VIENE MATRICE = []
        self.N = 0                                                  # N - side of the lattice 
        self.lattice = []                                           # lattice
        self._J = None                                               # weight matrix 
        self.C = None                                               # connectivity matrix 
        self.B = None                                               # bias matrix 
        self.D = None                                               # distance matrix - this is computed and stored for faster execution
        self.fitness = None                                         # Initialize it to None for avoiding recomputation 
        
    @property                                                       # I have to make it a property so that it changes dynamically with self.N
    def N_sol(self):
        return self.N **2                                           # N_sol = N**2 - Number of cells in the lattice (each site in the lattice has one cell in it)
    '''
    @property
    def lattice(self):                                              # lattice: N_sol - it's a string in which each spot S is a site in the actual 2D lattice (S = i * N + j)
        self._lattice = [None] * self.N_sol                         # I have to use list bc of JAX. this creates an N_sol list
        return self._lattice
    @lattice.setter
    def lattice(self,value):
        self._lattice = value
    '''
    @property
    def J(self):                                                    
        self._J = jnp.ones((self.N_sol,self.N_sol))
        return self._J
    @J.setter
    def J(self, value):
        self._J = value
    @property
    def C(self):                                                    
        self._C = jnp.ones((self.N_sol,self.N_sol))
        return self._C
    @C.setter
    def C(self, value):
        self._C = value
    @property
    def B(self):                                                    
        self.B = jnp.zeros((self.N_sol,self.N_sol))
        return self._B
    @B.setter
    def B(self, value):
        self._B = value
    @property
    def D(self):                                                    
        self._D = jnp.zeros((self.N_sol,self.N_sol))
        return self._D
    @D.setter
    def D(self, value):
        self._D = value
    
    def generate(self,par):
        # First set the parameters of the Network from par
        self.N = par['N']
        # First generate a cell for each lattice site and assign the former to the latter
        for s in range(self.N_sol):
            cell = Cell()                                       # Initialize a Cell object
            cell.generate(par)                                  # Generate it - (also par['key] gets updated here)
            cell.S = s                                          # Assign a new attribute 'S' which is the site in the 1D lattice string
            cell.coord = (s // self.N, s % self.N)              # Assign a new attribute 'coord' to the cell object and set it to the coordinates of the lattice site
            self.lattice.append(cell)                         # Assign the generated cell to the lattice site
        jdb.print('Lattice ready.')
        # Then randomly generate weight, bias and connectivity matrices
        subkey, par = gen_key(par)                                      # generate the subkey for the random generation in the following line
        jdb.print('{s}',s=self.J.shape)
        self.J = jrd.uniform(subkey,shape=self.J.shape)                 # uniformly populate the weights in the weight matrix J 
        jdb.print('{j}',j=self.J)     
        subkey, par = gen_key(par)                                      # generate the subkey for the random generation in the following line
        self.B = jrd.normal(subkey,shape=self.B.shape)                  # extract from a normal distribution centered in 0 with st. dv. = 1 the biases (spero vada bene fatto così)
        self.D = cdist(jnp.array([cell.coord for cell in self.lattice]),       # DA CONTROLLARE!!
                       jnp.array([cell.coord for cell in self.lattice]),vmap=True) # compute the distance between each two cells, given their coordinates
        subkey, par = gen_key(par)
        prob_matrix = jnp.where(jnp.eye(self.N_sol, dtype=bool),        # define a matrix for the link probability for C
                    par['p_self_link'], 1 / self.D)
        self.C = jnp.where(jrd.bernoulli(subkey, prob_matrix),1,0)      
        jdb.print('Matrices ready.')
        # Already compute the fitness of the network
        #self.compute_fitness(par)
    
    # QUI LA FITNESS è PARI PARI A QUELLA DELLO XOR, LA DEVO RIDEFINIRE
    def compute_fitness(self, par):
        if self.fitness == None:        # I need this check, bc otherwise I risk adding fitness over fitness
            self.fitness = 0.           # This is to avoid type conflict and to make sure that I'm not computing the fitness of a network that already has it
            for i,input in enumerate(par['input_set']):
                jdb.print('Input #{i}',i=i)
                output = self.ff(input)
                squared_dist = (par['target_set'][i] - output)**2     # square distance between network output and target (theoretical) output
                cost = (jnp.sum(self.C.flatten())/2)      # here the cost is computed only on the presence or absence of links
                self.fitness += 50*squared_dist + cost # add to the fitness value of the network
            self.fitness /= len(par['input_set'])    # normalize over the inputs
    
    def ff(self, input:list, verb:int=0):
        # set the value of the two inputs cells through the input value
        self.lattice[0].value = input[0]                        # cell in the upper left corner of the 2D lattice
        self.lattice[(self.N - 1) * self.N-1].value = input[1]    # cell in the lower left corner of the 2D lattice
        for s in range(self.N_sol):
            cell = self.lattice[s]
            inbound = 0     # value to give in input to the cell, which then computes its value through the activation
            for b in range(self.N_sol):             # here we're looping over the COLUMNS of the matrices, bc the element (a,b) of J is the link FROM b TO a
                # note: here we're treating self links as any other link
                inbound += self.C[s,b] * self.J[s,b] * self.lattice[b].value + self.B[s,b]      # weight x value + bias
                #jdb.print('{a}',a=self.C[s,b] * self.J[s,b] * self.lattice[b].value + self.B[s,b])
            cell.compute(inbound)           # pass all the contributions through the activation function to determine the value of the cell
        if verb > 0: 
            return [c.value for c in self.lattice]
        output = self.lattice[self.N_sol-1].value         # the cell in the right lower corner is the output
        return output
            

In [39]:
subkey, par = gen_key(par)
a = jrd.uniform(subkey,(2,2))
a

Array([[0.6721586 , 0.34927034],
       [0.6755668 , 0.6915271 ]], dtype=float32)

In [41]:
n = Network()
n.generate(par)
#jdb.print('{a}',a=[n.lattice[c].value for c in range(n.N_sol)])
jdb.print('{a}',a=n.J)
# set the value of the two inputs cells through the input value
def ff(input,verb:int=0):
    n.lattice[0].value = input[0]                        # cell in the upper left corner of the 2D lattice
    n.lattice[(n.N - 1) * n.N-1].value = input[1]    # cell in the lower left corner of the 2D lattice
    for s in range(n.N_sol):
        cell = n.lattice[s]
        # here we're looping over the COLUMNS of the matrices, bc the element (a,b) of J is the link FROM b TO a
        # note: here we're treating n links as any other link
        a = jnp.array([n.C[s,b] * n.J[s,b] * n.lattice[b].value + n.B[s,b] for b in range(n.N_sol)])
        jdb.print('{a}',a=a[0:10])
        jdb.print('{a}',a=n.C[s,0])
        jdb.print('{a}',a=n.J[s,0])
        jdb.print('{a}',a=n.lattice[0].value)
        jdb.print('{a}',a=n.B[s,0])
        inbound = jnp.sum(jnp.array([n.C[s,b] * n.J[s,b] * n.lattice[b].value + n.B[s,b] for b in range(n.N_sol)]),axis=1)      # weight x value + bias
        cell.compute(inbound)           # pass all the contributions through the activation function to determine the value of the cell
    if verb > 0: 
        return [c.value for c in n.lattice]
    output = n.lattice[n.N_sol-1].value         # the cell in the right lower corner is the output
    return output

#output1 = n.ff(par['input_set'][0])
#print(output1)
output0 = ff(par['input_set'][0])
print(output0)



Lattice ready.
(Array(100, dtype=int32, weak_type=True), Array(100, dtype=int32, weak_type=True))
[[1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]
 ...
 [1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]]
Matrices ready.
[[1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]
 ...
 [1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]]
[0. 0. 0. 1. 0. 1. 1. 1. 0. 0.]
1.0
1.0
0
0.0


ValueError: axis 1 is out of bounds for array of dimension 1

In [5]:
n = Network()
n.generate(par)
jdb.print('{B}',B=n.B)

Lattice ready.
Matrices ready.
Input #0
Input #1
Input #2
Input #3
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [6]:
a = []
for i in range(5):
    a.append(i)
print(len(a))
a[5]

5


IndexError: list index out of range

In [ ]:
n.lattice[0].coord
c=jnp.array([cell.coord for cell in n.lattice])
print(c.shape)
print(c[0])
print(cdist(c[0],c[1]))
print(cdist(c,c,True))
subkey, par = gen_key(par)
print(jrd.choice(subkey,jnp.arange(2),p=jnp.array([0.5,0.5])))

In [ ]:
# Execution
# Notes: put N as a static argname in .jit()
# REMEMBER TO DEAL WITH THE BOUNDARY CONDITIONS ON THE LATTICE